# Huấn luyện LTX 2.3 LoRA với Colab Notebook

Notebook này được tạo ra để hướng dẫn huấn luyện LoRA cho mô hình LTX 2.3 sử dụng thư viện `musubi-tuner`.

**Chú ý**: Quá trình huấn luyện đòi hỏi khá nhiều VRAM (khuyến nghị GPU có 24GB VRAM trở lên đối với huấn luyện video). Nếu bạn dùng Colab, hãy đảm bảo chọn GPU A100 hoặc L4/T4 (kèm tối ưu hóa mạnh như phân bổ khối swap, tile VAE, v.v.).


## Bước 1: Chuẩn bị Môi trường

Cài đặt thư viện và tải xuống các dependency cần thiết.

In [ ]:
!git clone https://github.com/kohya-ss/musubi-tuner.git
%cd musubi-tuner

# Cài đặt PyTorch phù hợp với hệ thống Colab (Thường là CUDA 12.1 hoặc 12.2)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Cài đặt musubi-tuner và các thư viện cần thiết
!pip install -e .
!pip install ascii-magic matplotlib tensorboard accelerate huggingface_hub

## Bước 2: Tải Mô hình (LTX 2.3 & Gemma)

LTX 2.3 yêu cầu mô hình chính (checkpoint) và Gemma 3 12B làm text encoder.
Chúng ta sẽ tải phiên bản LTX 2.3 từ `Lightricks/LTX-2.3` và Gemma FP8 safetensors từ `GitMylo/LTX-2-comfy_gemma_fp8_e4m3fn`.

In [ ]:
import os
from huggingface_hub import hf_hub_download

MODEL_DIR = "/content/models"
os.makedirs(MODEL_DIR, exist_ok=True)

print("Đang tải LTX 2.3 Checkpoint...")
ltx2_ckpt = hf_hub_download(repo_id="Lightricks/LTX-2.3", filename="ltx-2.3-22b-dev.safetensors", local_dir=MODEL_DIR)

print("Đang tải Gemma FP8 Text Encoder...")
gemma_ckpt = hf_hub_download(repo_id="GitMylo/LTX-2-comfy_gemma_fp8_e4m3fn", filename="gemma_3_12B_it_fp8_e4m3fn.safetensors", local_dir=MODEL_DIR)

print(f"LTX 2.3 lưu tại: {ltx2_ckpt}")
print(f"Gemma lưu tại: {gemma_ckpt}")

## Bước 3: Tạo Dataset và Cấu hình TOML

Dưới đây là đoạn code mẫu tạo ra cấu trúc thư mục chứa dataset và file cấu hình `dataset.toml`.

In [ ]:
import os

DATA_DIR = "/content/dataset/videos"
CACHE_DIR = "/content/dataset/cache"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# [TUỲ CHỌN] Bạn có thể tải video/image lên thư mục /content/dataset/videos tại đây.
# Mỗi video (vd: video1.mp4) cần có file caption đi kèm (vd: video1.txt).

toml_content = f"""[general]
resolution = [768, 512]
caption_extension = ".txt"
batch_size = 1
enable_bucket = true

[[datasets]]
video_directory = "{DATA_DIR}"
cache_directory = "{CACHE_DIR}"
target_frames = [1, 17, 33]
target_fps = 25
"""

with open("dataset.toml", "w") as f:
    f.write(toml_content)
    
print("Đã tạo dataset.toml thành công.")
print("Hãy tải video và caption của bạn vào thư mục /content/dataset/videos trước khi chạy các bước tiếp theo!")

## Bước 4: Pre-caching (Latents & Text Encoder)

Để tiết kiệm VRAM và tăng tốc độ huấn luyện, chúng ta sẽ mã hóa trước latents của video/image và outputs của text encoder.

In [ ]:
# 1. Cache Latents
!(python ltx2_cache_latents.py \
  --dataset_config dataset.toml \
  --ltx2_checkpoint /content/models/ltx-2.3-22b-dev.safetensors \
  --device cuda \
  --vae_dtype bf16 \
  --vae_chunk_size 16 \
  --ltx2_mode video)

In [ ]:
# 2. Cache Text Encoder Outputs
!(python ltx2_cache_text_encoder_outputs.py \
  --dataset_config dataset.toml \
  --ltx2_checkpoint /content/models/ltx-2.3-22b-dev.safetensors \
  --gemma_safetensors /content/models/gemma_3_12B_it_fp8_e4m3fn.safetensors \
  --device cuda \
  --mixed_precision bf16 \
  --ltx2_mode video \
  --batch_size 1)

## Bước 5: Huấn luyện LoRA

Chạy lệnh sau để bắt đầu quá trình huấn luyện bằng `accelerate`. Chúng ta sử dụng FP8 và các phương thức tối ưu hóa bộ nhớ cho Colab (VRAM 16-24GB).

In [ ]:
!accelerate launch --num_cpu_threads_per_process 1 --mixed_precision bf16 ltx2_train_network.py \
  --mixed_precision bf16 \
  --dataset_config dataset.toml \
  --ltx2_checkpoint /content/models/ltx-2.3-22b-dev.safetensors \
  --ltx_version 2.3 \
  --ltx_version_check_mode warn \
  --ltx2_mode video \
  --fp8_base --fp8_scaled \
  --blocks_to_swap 30 \
  --use_pinned_memory_for_block_swap \
  --gradient_checkpointing \
  --gradient_checkpointing_cpu_offload \
  --sdpa \
  --learning_rate 1e-4 \
  --network_module networks.lora_ltx2 \
  --network_dim 32 \
  --network_alpha 32 \
  --timestep_sampling shifted_logit_normal \
  --output_dir /content/output \
  --output_name ltx23_lora \
  --max_train_epochs 10 \
  --save_every_n_epochs 5

Sau khi hoàn thành, file LoRA (`ltx23_lora.safetensors` và `ltx23_lora.comfy.safetensors`) sẽ được lưu tại thư mục `/content/output`.